# HW2 Starter：CNN 醫學影像分類

**對應單元**：W6 CNN / 遷移學習　|　**環境**：Google Colab（建議開 T4 GPU）

> 這是骨架，填完每個 `# TODO` 即可。刻意設計成 Colab 免費層可跑完：小圖、小模型、少 epoch。

## 環境設定

In [ ]:
!pip install -q medmnist
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt
from medmnist import PneumoniaMNIST, INFO
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', dev)
# 沿用 HW1：from mypreproc import normalize   # 上傳 mypreproc.py 後 import

## B1 載入與檢視
載入資料、印筆數/類別分布、顯示每類範例。

In [ ]:
train = PneumoniaMNIST(split='train', download=True, size=28)
val   = PneumoniaMNIST(split='val',   download=True, size=28)
test  = PneumoniaMNIST(split='test',  download=True, size=28)
# TODO: 印出 train/val/test 筆數、影像 shape、類別分布（是否不平衡？）
# TODO: 每類顯示幾張範例圖

## B2 前處理與 DataLoader
normalization + augmentation + DataLoader。

In [ ]:
from torchvision import transforms
from torch.utils.data import DataLoader, TensorDataset
# TODO: 用 HW1 mypreproc 或這裡做 normalize；加 augmentation：小角度旋轉、亮度擾動；不可上下翻轉（10/12 augmentation 頁）
# 提示：transform 可直接傳給 PneumoniaMNIST(split=..., transform=..., download=True, size=28)
# 提示：labels 形狀是 (N,1)，送進 CrossEntropyLoss 前要 y.squeeze().long()
# TODO: 建 train/val/test DataLoader（batch_size 如 64）

## B3 小 CNN
2-3 個卷積區塊 + GAP + 線性頭，參數 < 1M。

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_cls=2):
        super().__init__()
        # TODO: 卷積區塊(Conv-BN-ReLU-Pool) x2~3 -> AdaptiveAvgPool2d(1) -> Linear
        pass
    def forward(self, x):
        # TODO
        return x
model = SmallCNN().to(dev)
print('params:', sum(p.numel() for p in model.parameters()))

## B4 訓練
交叉熵 + Adam，≤10 epoch，畫 loss/acc 曲線。

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()
# TODO: 訓練迴圈（≤10 epoch），記錄 train/val loss 與 accuracy
# TODO: 畫曲線；觀察 val 是否回升（過擬合）

# 訓完存權重到 Drive（HW5 要載回；路徑自行調整）
# torch.save(model.state_dict(), '/content/drive/MyDrive/medimg_toolkit/hw2_cnn.pt')

## B5 誠實評估
AUROC + 混淆矩陣 + P/R/F1。把 AUROC 寫進 mymetrics.py。

In [ ]:
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
# TODO: 在 test 上收集機率與標籤，算 AUROC、混淆矩陣、precision/recall/F1
# TODO: 畫混淆矩陣；比較 accuracy vs AUROC 說的故事是否一致

## B6 CAM
2-3 張測試圖畫 CAM 疊圖（W6 的類別活化圖；你的 SmallCNN 是 GAP+線性頭，直接適用）。

In [ ]:
# CAM（GAP+線性頭專用，比 Grad-CAM 簡單）：
#   1. 取「送進 GAP 的那個張量」A (shape [C,h,w])：最後一個卷積區塊的輸出（Conv-BN-ReLU-Pool 的話是 Pool 之後）。可對 GAP 層掛 forward hook，取它的 input[0]
#      自我檢查：A.mean((1,2)) @ w_c + b_c 應等於模型輸出的 logit_c
#   2. 取分類頭 Linear 對「預測類別 c」那一列權重 w_c (shape [C])
#   3. CAM = ReLU( sum_k w_c[k] * A[k] )，上採樣回原圖大小、正規化到 [0,1]、疊圖
# TODO: 依上面三步實作 CAM，對 2-3 張測試圖畫疊圖
# 觀察：模型看的是病灶還是別處？（連 10/12 CAM 頁）
# 進階(加分)：改用 Grad-CAM（用梯度加權，11/23 那講）比較兩者差異

## Part D — 逆向工程（繳交介面固定）

助教會用**你沒看過的保留測例**呼叫下面兩個函式，所以：

- 函式名稱、參數順序、回傳格式 **不可更動**：D1 回傳 (N,28,28) 的 `np.float32`；D2 回傳 (state_dict, 長度 21 的 loss 陣列)。
- 不要讀你自己另存的檔，也不要用全域變數存推出來的參數，常數請寫死在函式內。例外：D1 可以使用載入格建立的教師模型（`TeacherCNN` 類別與 `cnn_state`），助教評分時會用同名變數、在自己的 REVERSE 路徑下提供。
- 函式要能重複呼叫，不依賴前面 cell 的執行順序。
- 報告要寫推理過程：至少比較兩個假說、附支持的圖或數字、說明哪裡不可辨識。

門檻與檔案格式見 `reverse/manifest.json`。下面的自評格只檢查公開資料，通過不代表保留測例一定過。

In [ ]:
# === Part D 資料載入 ===
# Colab：把課程發的 reverse/ 整個資料夾放進自己的 Drive 再掛載，或用 files.upload() 上傳。
# from google.colab import drive; drive.mount('/content/drive')
import os, json
import numpy as np

REVERSE = '/content/drive/MyDrive/HW2/reverse'   # ← 改成你的路徑
man = json.load(open(os.path.join(REVERSE, 'manifest.json'), encoding='utf-8'))
def L(name): return np.load(os.path.join(REVERSE, name))
def LL(names): return [np.load(os.path.join(REVERSE, x)) for x in names]
import csv, sys
import torch
sys.path.append(REVERSE)                      # teacher_cnn.py 在 reverse/ 裡

D1_in = np.stack(LL(man['D1']['files']['inputs']))       # (8,28,28)
D1_tg = np.stack(LL(man['D1']['files']['targets']))      # (8,28,28)
D1_labels = L('D1_labels.npy')
cnn_state = torch.load(os.path.join(REVERSE, 'cnn.pt'), map_location='cpu')
# from teacher_cnn import <類別名>          # 類別名見 teacher_cnn.py

D2_feat = L('D2_features.npy')                # (64,16)
D2_lab = L('D2_labels.npy')                   # (64,)
D2_head_init = torch.load(os.path.join(REVERSE, 'D2_head_init.pt'), map_location='cpu')
D2_head_final = torch.load(os.path.join(REVERSE, 'D2_head_final.pt'), map_location='cpu')
D2_loss = np.array([float(r['loss']) for r in
                    csv.DictReader(open(os.path.join(REVERSE, 'D2_loss.csv'), encoding='utf-8'))])
print('D1', D1_in.shape, '| D2 feat', D2_feat.shape, 'loss pts', len(D2_loss))

In [ ]:
# === D1 這個 CAM 是怎麼畫出來的 ===
from skimage.metrics import structural_similarity as ssim_fn

def report(rows):
    for i, (txt, ok) in enumerate(rows):
        print(f'[{i}] {txt}  ' + ('PASS' if ok else 'FAIL'))
    print('== ALL PASS ==' if all(ok for _, ok in rows) else '== 尚未達標 ==')

def reproduce_D1(inputs):
    # inputs: np.ndarray (N,28,28) float32 -> np.ndarray (N,28,28) float32
    # 要自己載入 cnn.pt 前向取 16x14x14 特徵圖與 fc 權重；目標類別 / fc bias /
    # 負值處理 / 正規化方式 / 正規化與 resize 的先後，都是你要推的設計選擇
    inputs = np.asarray(inputs, dtype=np.float32)
    # TODO: 重建 CAM 繪製流程
    return np.zeros_like(inputs, dtype=np.float32)

# --- 自評：每張 MAE <= 0.003 且 SSIM >= 0.98 ---
pred = reproduce_D1(D1_in)
rows = []
for p, t in zip(pred, D1_tg):
    m = float(np.abs(t - p).mean()); s = ssim_fn(t, p, data_range=1.0)
    rows.append((f'MAE={m:.5f} SSIM={s:.5f}', m <= 0.003 and s >= 0.98))
report(rows)

In [ ]:
# === D2 從權重更新反推訓練配方 ===
def reproduce_D2(features, labels, head_init_state_dict):
    # -> (final_state_dict, losses)；losses 長度 21（含第 0 步）
    # 只訓練 nn.Linear(16,2)，每步都用同一個 full batch。要推的是 loss 是否 class-weighted、
    # optimizer 的 lr / momentum（更新次數已知為 20）
    import torch, torch.nn as nn
    head = nn.Linear(16, 2)
    head.load_state_dict({k: v.clone() for k, v in head_init_state_dict.items()})
    X = torch.as_tensor(features, dtype=torch.float32)
    y = torch.as_tensor(labels, dtype=torch.long)
    losses = []
    # TODO: 依你推出的 loss / optimizer 設定跑 20 步，記錄 21 個 loss
    return head.state_dict(), np.array(losses, dtype=np.float64)

# --- 自評：loss 曲線 RMSE <= 1e-4 且參數最大誤差 <= 1e-3 ---
sd, curve = reproduce_D2(D2_feat, D2_lab, D2_head_init)
rmse = float(np.sqrt(np.mean((curve - D2_loss) ** 2))) if len(curve) == len(D2_loss) else float('inf')
perr = max(float((sd[k] - D2_head_final[k]).abs().max()) for k in D2_head_final)
report([(f'curve RMSE={rmse:.2e}', rmse <= 1e-4), (f'max |dparam|={perr:.2e}', perr <= 1e-3)])